# Face-aging baseline comparison

This notebook uses the project comparison API. The loader prepares SAM, FRAN, FADING, CUSP and HRFAE sequentially and applies the required FADING/HRFAE compatibility patches internally. Install the comparison repositories' dependencies in the server environment before running it.

In [ ]:
from pathlib import Path

from src.comparisong import age_image, load_aging_models

## Paths and benchmark selection

Set `FADING_SPECIALIZED_PATH` when the checkpoint already exists in a custom location. Leaving it as `None` preserves the loader's existing discovery/download behavior when you run this notebook on the server.

In [ ]:
BENCHMARK_ROOT = Path("/content/aging_benchmarks")
SOURCE_IMAGE = Path("/content/drive/MyDrive/dataset_unificado/id_100/26.jpg")
OUTPUT_DIR = Path("/content/aging_outputs/persona_004")
FADING_SPECIALIZED_PATH = None
EVALUATOR_ROOT = Path("/content/evaluation_models")

MODELS = ("SAM", "FRAN", "FADING", "CUSP", "HRFAE")
SOURCE_AGE = 26
TARGET_AGES = [8, 16, 26, 45, 55, 65, 75]
GENDER = "female"

# Real same-person longitudinal targets. Partial mappings are valid.
TARGET_IMAGES = {
    45: Path("/content/drive/MyDrive/dataset_unificado/id_100/45.jpg"),
    65: Path("/content/drive/MyDrive/dataset_unificado/id_100/65.jpg"),
}
METRICS_CONFIG = {
    "adaface_repo_path": EVALUATOR_ROOT / "AdaFace",
    "adaface_checkpoint_path": EVALUATOR_ROOT / "adaface_ir101_webface12m.ckpt",
    "dex_prototxt_path": EVALUATOR_ROOT / "DEX/age.prototxt",
    "dex_checkpoint_path": EVALUATOR_ROOT / "DEX/dex_chalearn_iccv2015.caffemodel",
    "kid_inception_weights_path": EVALUATOR_ROOT / "KID/weights-inception-2015-12-05-6726825d.pth",
    "device": "auto",
}

## 1. Load the comparison models

Models stay on CPU whenever their implementation permits it and are moved or executed sequentially during inference to control VRAM usage.

In [ ]:
bundle = load_aging_models(
    root=BENCHMARK_ROOT,
    models=MODELS,
    fading_specialized_path=FADING_SPECIALIZED_PATH,
    cusp_variant="ffhq_ls",
    download_cusp=True,
    download_hrfae=True,
    download_cradle=False,
    load_metrics=True,
    metrics_config=METRICS_CONFIG,
)

## 2. Generate the comparison sweep

Every available baseline writes age-specific images and one horizontal sweep under `OUTPUT_DIR`. With `strict=False`, an unavailable legacy baseline is reported without stopping the remaining models.

In [ ]:
comparison = age_image(
    bundle=bundle,
    image=SOURCE_IMAGE,
    source_age=SOURCE_AGE,
    target_ages=TARGET_AGES,
    gender=GENDER,
    models=MODELS,
    seed=42,
    show=True,
    output_dir=OUTPUT_DIR,
    strict=False,
    metrics=True,
    target_images=TARGET_IMAGES,
    metrics_output_dir=OUTPUT_DIR / "metrics",
    evaluation_config={"kid_subsets": 100},
)

display(comparison["metrics"]["summary"])
print(comparison["metrics"]["paths"])